# Harry Potter RAG — Lab 3: Evaluation

Continuing from [Lab 2: Evaluation Prep](./lab2-evaluation-prep.ipynb), which produced
the labelled set at `labs/eval/qa_candidates.jsonl`.

This lab measures the pipeline from Lab 1: it loads the labelled questions, runs the
retrieval and generation stages, and puts numbers behind every design choice.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Goal of Lab 3: RAG Evaluation & Ablation Matrix

We take the baseline pipeline and measure how different design choices actually move
the numbers. Instead of treating RAG like a black box, we split evaluation into two
separate questions:

1. Did we retrieve the right context?
2. Did the LLM generate a correct, grounded answer?

### What we're testing

1. **Ground-truth evaluation** — load the labelled Q&A set: questions with exact
   evidence snippets, multi-hop questions, and out-of-scope probes that test
   refusal behaviour.

2. **Retriever comparison** — dense (Titan V2), sparse (BM25) and hybrid
   (Reciprocal Rank Fusion) over the same chunk index.

3. **Reranker** — reorder a top-50 candidate pool with a Bedrock reranker before
   the top-5 reaches the model.

4. **Separating the failures**
    - **Retriever metrics:** `Hit@k`, `Recall@k` and `MRR` — did the labelled evidence make the context window?
    - **Generator metrics:** an LLM judge for correctness and faithfulness, plus abstention on answerable questions and on out-of-scope probes.
    - **Closed-book control:** the same questions with retrieval off, to separate what the model already knows from what the context supplies.

5. **Results leaderboard** — every run logs to `eval/*.csv`, so architectural
   choices are backed by numbers rather than vibes.

**A chunk-size ablation was tried and dropped.** A sentence-packed 256-token index
was built and measured against the 512-token baseline; it was worse on every
retrieval metric (`Hit@5` 0.83-0.86 vs 0.908), so the chunking matrix was removed
to keep the notebook focused. The reranker is the ablation that moved the numbers.

## Preconditions

- **Lab 1 pipeline** — embeddings, the Chroma collection, BM25, and the `ask()` loop.
- **Lab 2 output** — `labs/eval/qa_candidates.jsonl`. Filter on `locatable` and treat
  `evidence_chunk_ids` as the retrieval label. Lab 3 never regenerates it.

## Build order

1. Load the labelled set and the chunk index.
2. Compare retrievers, then rerank a top-50 candidate pool before the top-5.
3. Retrieve, then score `Hit@k` / `Recall@k` / `MRR` against `evidence_chunk_ids`.
4. Score generation: `EM` / `F1`, plus a judge for correctness and faithfulness.
5. Re-run generation closed-book to measure how much of the score is memory.
6. Emit one row per question to `labs/eval/results.csv`.

## Import dependencies

In [1]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import csv
import collections

/Users/tc/Documents/Github/monitus_code/labs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reusing functions from previous lab

In [2]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))

class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))

chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))

## Load the evaluation set

In [3]:
# Labelled evaluation set produced by Lab 2 (Evaluation Prep).

def repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p


EVAL = repo_root() / "labs" / "eval"
rows = [json.loads(line) for line in open(EVAL / "qa_candidates.jsonl", encoding="utf-8") if line.strip()]
usable = [r for r in rows if r.get("locatable")]

print(f"{len(rows)} candidates | {len(usable)} usable (locatable)")
print("types:", dict(collections.Counter(r["type"] for r in usable)))
print("chapters:", len({r["chapter"] for r in usable}))
usable[0]

69 candidates | 65 usable (locatable)
types: {'factoid': 62, 'multi-hop': 3}
chapters: 17


{'id': 'cand-hp1-01-01',
 'book': 1,
 'chapter': 1,
 'chapter_name': 'Chapter 1 - The Boy Who Lived',
 'type': 'factoid',
 'question': 'What is the name of the firm where Mr. Dursley works?',
 'answer': 'Grunnings',
 'evidence': 'Mr. Dursley was the director of a firm called Grunnings, which made drills.',
 'evidence_chunk_ids': ['hp1-ch01-0001'],
 'n_spans': 1,
 'locatable': True,
 'distinctive': True,
 'unmatched_spans': [],
 'status': 'draft'}

## Rebuild the retrieval schema

Lab 3 runs on its own rather than importing from Lab 1. 

The chunks, the Chroma collection and the BM25 index are re-attached here

Therefore, the collection name and `INDEX_CONFIG` must match the ones used at ingest, because the retrieval labels in `qa_candidates.jsonl` are ids from that index.

In [4]:
INDEX_CONFIG = {
    "embedding_model": "amazon.titan-embed-text-v2:0",
    "dimensions": 1024,
    "normalize": True,
    "chunk_size": 512,
    "overlap_ratio": 0.15,
    "tokenizer": "o200k_base",
    "chunking": "sliding_window",
}


class BedrockOnly:
    """Sentinel embedding function: every vector must come from Bedrock."""

    def __call__(self, input):
        raise RuntimeError("Chroma must not embed; pass embeddings= from Bedrock.")

    def embed_query(self, input):
        raise RuntimeError("Use query_embeddings= with Bedrock vectors.")

    def name(self):
        return "bedrock_only"


def get_collection(client, name, config=INDEX_CONFIG):
    client.get_or_create_collection(
        name=name,
        embedding_function=BedrockOnly(),
        configuration={"hnsw": {"space": "cosine"}},   # distance = 1 - cos_sim
        metadata=config,
    )
    return client.get_collection(name, embedding_function=BedrockOnly())


COLLECTION_NAME = "hp1-c512-ov15"
collection = get_collection(chroma_client, name=COLLECTION_NAME)
collection.count()

249

## Load the chunk index and build the lexical index

Lexical index: "maps individual words, terms, or tokens directly to their locations within a document"

Chroma: has vector + metadata + id

BM25 is built on raw chunk + metadata

In [5]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


CHUNKS = read_jsonl(DATA / "harrypotter-chunks.jsonl")
CHUNK_BY_ID = {c["chunk_id"]: c for c in CHUNKS}

# Match length between chunk ids vs actual chunk to avoid mismatch
assert len(CHUNK_BY_ID) == len(CHUNKS), "duplicate chunk ids in chunk index"
assert collection.count() == len(CHUNKS), (
    f"collection has {collection.count()} vectors but index has {len(CHUNKS)} chunks"
)


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


# Lexical index over the exact same chunks that were embedded
bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])

print(f"{len(CHUNKS)} chunks | {collection.count()} vectors | {len(bm25.idf)} BM25 terms")

249 chunks | 249 vectors | 5737 BM25 terms


## Three retrievers behind one interface

All three return ordered chunk ids. 

Scoring (and ranking) is done on **ids**, not by direct strings.

### Reciprocal Rank Fusion (RRF)

The hybrid retriever fuses two rankings whose scores are not comparable: dense
returns a **cosine distance** (0-2, smaller is better) and BM25 returns an
unbounded **term-weight sum** (larger is better). Adding them or normalising them
is fragile. RRF sidesteps the problem by discarding the scores and using only
**rank position**:

$$
\mathrm{score}(d) = \sum_{r \in \{\text{dense},\ \text{sparse}\}} \frac{1}{k + \mathrm{rank}_r(d)}
$$

Every chunk receives a contribution from each list it appears in, so a chunk
retrieved by **both** retrievers accumulates credit. The top-k of the fused
ranking is returned.

- `rrf_k = 60` (the damping constant, from Cormack et al. 2009) flattens the
  curve: ranks 1, 2 and 3 contribute `1/61`, `1/62` and `1/63`, so a single
  rank-1 hit cannot dominate, while agreement across both lists still wins.
- This is **not** the same `k` as `top_k`. `top_k` controls how many results are
  returned; `rrf_k` is a fixed constant and belongs in the repro record as the
  **fusion constant**.

Worked example from the smoke run below:

| chunk | dense rank | sparse rank | fused score (`rrf_k = 60`) |
|---|---|---|---|
| `hp1-ch01-0001` | 1 | 1 | `2/61 = 0.0328` |
| `hp1-ch01-0003` | 2 | 3 | `1/62 + 1/63 = 0.0320` |
| `hp1-ch01-0006` | 3 | 2 | `1/63 + 1/62 = 0.0320` |

`hp1-ch01-0001` wins because both retrievers agree on it; the other two tie and
keep insertion order.

In [6]:
def dense_ranking(question_embedding, k):
    """Cosine top-k from the Chroma collection."""
    return collection.query(
        query_embeddings=question_embedding,
        n_results=k,
        include=[],
    )["ids"][0]


def sparse_ranking(question, k):
    """BM25 top-k; argsort maps a score position back to its chunk."""
    scores = bm25.get_scores(tokenize(question))
    return [CHUNKS[i]["chunk_id"] for i in np.argsort(scores)[::-1][:k]]


def hybrid_ranking(question_embedding, question, k, rrf_k=60):
    """Reciprocal Rank Fusion: score = sum(1 / (rrf_k + rank)) over both rankings."""
    fused = {}
    for ranking in (dense_ranking(question_embedding, k), sparse_ranking(question, k)):
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
    return [cid for cid, _ in sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]]



| Retriever | Uses | Ranking basis |
|---|---|---|
| `dense` | `embedding` only | cosine distance between query vector and chunk vectors |
| `sparse` | `question` only | BM25 term overlap after `tokenize(question)` |
| `hybrid` | both | RRF over the two rankings |


In [7]:
# Uniform call shape: (question, question_embedding, k) -> ranked chunk ids
RETRIEVERS = {
    "dense": lambda question, embedding, k: dense_ranking(embedding, k),
    "sparse": lambda question, embedding, k: sparse_ranking(question, k),
    "hybrid": lambda question, embedding, k: hybrid_ranking(embedding, question, k),
}

smoke = usable[0]
for name, fn in RETRIEVERS.items():
    print(f"{name:>6}: {fn(smoke['question'], bedrock_client.embed(smoke['question']), 3)}")
print("expected evidence:", smoke["evidence_chunk_ids"])

 dense: ['hp1-ch01-0001', 'hp1-ch01-0003', 'hp1-ch01-0006']


sparse: ['hp1-ch01-0001', 'hp1-ch01-0006', 'hp1-ch01-0003']


hybrid: ['hp1-ch01-0001', 'hp1-ch01-0003', 'hp1-ch01-0006']
expected evidence: ['hp1-ch01-0001']


## Score retrieval against the evidence labels

`Hit@k`, `Recall@k` and `MRR` are computed per question, then averaged. This is a
pure retrieval measurement — no generation is involved yet.

- `Hit@k` — how often at least one labelled chunk made the window.
- `Recall@k` — how much of the labelled evidence made it; 13 of the 65 questions
  carry more than one relevant chunk, so this can be below `Hit@k`.
- `MRR` — how high the first relevant chunk sat.

`top_k` and the fusion constant (`rrf_k = 60`) are repro knobs: record them with
any number you publish. Next comes the config matrix (chunk size / chunking
strategy, which requires rebuilding indices and re-locating the evidence spans),
then the generation half — `EM` / `F1`, the judge, and abstention.

In [8]:
def retrieval_scores(retrieved_ids, relevant_ids):
    """Hit@k, Recall@k and reciprocal rank for one question."""
    relevant = set(relevant_ids)
    retrieved = list(retrieved_ids)
    hits = relevant.intersection(retrieved)
    rr = next(
        (1.0 / rank for rank, cid in enumerate(retrieved, start=1) if cid in relevant),
        0.0,
    )
    return {
        "hit": 1.0 if hits else 0.0,
        "recall": len(hits) / len(relevant) if relevant else 0.0,
        "rr": rr,
    }


def evaluate_retrieval(questions, k=5, retrievers=RETRIEVERS):
    """Score every retriever on the same questions and return mean metrics."""
    questions = list(questions)

    # Embed every question once, in one cached batch, and reuse the vector for
    # both dense and hybrid. The disk cache makes a re-run API-free.
    embeddings = bedrock_client.embed([r["question"] for r in questions])

    scores = {name: [] for name in retrievers}
    for record, embedding in zip(tqdm(questions, desc=f"retrieval@{k}"), embeddings):
        for name, fn in retrievers.items():
            scores[name].append(
                retrieval_scores(
                    fn(record["question"], embedding.reshape(1, -1), k),
                    record["evidence_chunk_ids"],
                )
            )

    table = []
    for name, rows_ in scores.items():
        n = len(rows_)
        table.append({
            "retriever": name,
            f"hit@{k}": sum(r["hit"] for r in rows_) / n,
            f"recall@{k}": sum(r["recall"] for r in rows_) / n,
            "mrr": sum(r["rr"] for r in rows_) / n,
        })
    return table


K = 5
results = evaluate_retrieval(usable, k=K)

header = f"{'retriever':<8}" + "".join(f"{col:>10}" for col in results[0] if col != "retriever")
print(header)
for row in results:
    print(f"{row['retriever']:<8}" + "".join(f"{row[col]:>10.3f}" for col in row if col != "retriever"))

retrieval@5:   0%|          | 0/65 [00:00<?, ?it/s]

retrieval@5:  57%|█████▋    | 37/65 [00:00<00:00, 367.31it/s]

retrieval@5: 100%|██████████| 65/65 [00:00<00:00, 398.90it/s]

retriever     hit@5  recall@5       mrr
dense        0.862     0.846     0.680
sparse       0.862     0.854     0.697
hybrid       0.908     0.900     0.742


### What the three scores say

| retriever | Hit@5 | Recall@5 | MRR |
|---|---|---|---|
| dense | 0.862 | 0.846 | 0.680 |
| sparse | 0.862 | 0.854 | 0.697 |
| hybrid | **0.908** | **0.900** | **0.742** |

- **Dense and sparse tie on Hit@5** at 56/65 questions, but they do not hit the
  *same* 56. The near-tie is the interesting part: two very different retrieval
  mechanisms reach the same coverage, which is what makes fusing them useful.
- **Sparse edges dense on ranking quality** - slightly higher Recall@5
  (0.854 vs 0.846) and MRR (0.697 vs 0.680). Chapter text is full of rare proper
  nouns, and an exact-term match places those chunks higher than an averaged
  embedding does.
- **Hybrid wins on all three.** Hit@5 rises to 59/65 (+4.6 points) and MRR to
  0.742, versus 0.680 (dense) and 0.697 (sparse): its first relevant chunk is
  ranked higher, not merely present. Where one retriever misses, the other often
  still has the chunk, and RRF promotes it.
- **Retrieval is not solved.** Roughly 9% of questions still have no labelled
  chunk in the top 5, and Recall@5 of 0.90 means a tenth of the evidence never
  reached the prompt. That is the headroom the next milestones attack - first
  with chunking and retriever ablations, then by separating retrieval misses
  from generation failures.

These are baseline numbers: one index (512-token chunks, 15% overlap), one
`top_k = 5`, and the Lab 2 label set. They are a fixed point to compare against,
not a final ranking of the retrievers.

## Retrieval failure analysis

Metrics say *how often* retrieval fails; this section says *why*. It takes every
question the hybrid retriever misses and asks three questions about it:

1. Where did the gold chunk actually rank in dense and in sparse, across the
   whole book?
2. Would it have been found if search were restricted to the question's own
   chapter? (A diagnostic, not a deployable filter - inference does not know the
   chapter.)
3. Does widening the fusion pool recover it?

In [9]:
def fuse(rankings, k, rrf_k=60):
    """RRF over precomputed rankings - the body of hybrid_ranking()."""
    fused = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
    return [cid for cid, _ in sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]]


# Dense side of the diagnostic, computed from the stored vectors rather than a
# deep collection.query(). Requesting n_results equal to the collection size can
# intermittently return one fewer id (Chroma HNSW), which silently drops a chunk
# and makes a deep-pool sweep flip between runs.
_stored = collection.get(include=["embeddings"])
DENSE_IDS = list(_stored["ids"])
DENSE_MAT = np.asarray(_stored["embeddings"], dtype=np.float32)
DENSE_MAT /= np.linalg.norm(DENSE_MAT, axis=1, keepdims=True)
assert len(DENSE_IDS) == DENSE_MAT.shape[0] == len(CHUNKS)


def full_rankings(question):
    """Deterministic full-book dense and sparse rankings, for diagnosis only."""
    query = bedrock_client.embed(question).reshape(-1).astype(np.float32)
    query /= np.linalg.norm(query)
    dense = [DENSE_IDS[i] for i in np.argsort(-(DENSE_MAT @ query))]
    scores = bm25.get_scores(tokenize(question))
    sparse = [CHUNKS[i]["chunk_id"] for i in np.argsort(scores)[::-1]]
    return dense, sparse


def first_rank(ranking, relevant):
    """1-based rank of the best relevant chunk, or None if absent."""
    return next((i for i, cid in enumerate(ranking, start=1) if cid in relevant), None)


# Full rankings are computed once and reused by every diagnostic below.
FULL = {r["id"]: full_rankings(r["question"]) for r in tqdm(usable, desc="full rankings")}

K = 5
failures = []
for record in usable:
    relevant = set(record["evidence_chunk_ids"])
    dense, sparse = FULL[record["id"]]
    retrieved = fuse([dense[:K], sparse[:K]], K)          # identical to hybrid_ranking()
    if relevant & set(retrieved):
        continue

    dense_ch = [c for c in dense if CHUNK_BY_ID[c]["chapter_num"] == record["chapter"]]
    sparse_ch = [c for c in sparse if CHUNK_BY_ID[c]["chapter_num"] == record["chapter"]]
    failures.append({
        "id": record["id"],
        "chapter": record["chapter"],
        "question": record["question"],
        "gold": sorted(relevant),
        "dense_rank": first_rank(dense, relevant),
        "sparse_rank": first_rank(sparse, relevant),
        "in_chapter_hit": bool(relevant & set(fuse([dense_ch[:K], sparse_ch[:K]], K))),
        "retrieved": retrieved,
    })

print(f"{len(failures)} hybrid misses out of {len(usable)}\n")
for f in failures:
    print(f"{f['id']} ch{f['chapter']}: {f['question']}")
    print(f"  gold {f['gold']}")
    print(f"  dense rank {f['dense_rank']} | sparse rank {f['sparse_rank']}"
          f" | recovered inside chapter: {f['in_chapter_hit']}")
    print(f"  retrieved: {f['retrieved']}")

recovered = sum(f["in_chapter_hit"] for f in failures)
chapter_hits = (len(usable) - len(failures)) + recovered
print(f"\nchapter-scoped hybrid Hit@{K}: {chapter_hits}/{len(usable)}"
      f" (global: {len(usable) - len(failures)}/{len(usable)})")

print("\nfusion pool depth vs global Hit@5:")
for pool in (K, 20, len(CHUNKS)):
    hits = sum(
        bool(set(r["evidence_chunk_ids"]) & set(fuse([FULL[r["id"]][0][:pool],
                                                      FULL[r["id"]][1][:pool]], K)))
        for r in usable
    )
    print(f"  pool={pool:>3}: {hits}/{len(usable)}")

full rankings:   0%|          | 0/65 [00:00<?, ?it/s]

full rankings:   2%|▏         | 1/65 [00:00<00:30,  2.13it/s]

full rankings:   3%|▎         | 2/65 [00:00<00:26,  2.37it/s]

full rankings:   5%|▍         | 3/65 [00:01<00:26,  2.32it/s]

full rankings:   6%|▌         | 4/65 [00:01<00:25,  2.41it/s]

...

full rankings:  97%|█████████▋| 63/65 [00:24<00:00,  2.65it/s]

full rankings:  98%|█████████▊| 64/65 [00:25<00:00,  2.66it/s]

full rankings: 100%|██████████| 65/65 [00:25<00:00,  2.65it/s]

full rankings: 100%|██████████| 65/65 [00:25<00:00,  2.54it/s]

6 hybrid misses out of 65

cand-hp1-01-04 ch1: What distinctive feature did Harry Potter have on his forehead?
  gold ['hp1-ch01-0013']
  dense rank 24 | sparse rank 27 | recovered inside chapter: True
  retrieved: ['hp1-ch15-0013', 'hp1-ch06-0008', 'hp1-ch17-0005', 'hp1-ch06-0009', 'hp1-ch05-0018']
cand-hp1-06-01 ch6: What name did Harry give to his new owl?
  gold ['hp1-ch06-0001']
  dense rank 59 | sparse rank 4 | recovered inside chapter: True
  retrieved: ['hp1-ch06-0010', 'hp1-ch05-0001', 'hp1-ch05-0017', 'hp1-ch04-0005', 'hp1-ch07-0002']
cand-hp1-10-03 ch10: What spell did Ron use to levitate the troll's club?
  gold ['hp1-ch10-0007']
  dense rank 12 | sparse rank 125 | recovered inside chapter: False
  retrieved: ['hp1-ch10-0011', 'hp1-ch10-0010', 'hp1-ch10-0009', 'hp1-ch16-0018', 'hp1-ch10-0012']
cand-hp1-11-03 ch11: What was the score of Harry's first Quidditch match?
  gold ['hp1-ch11-0010']
  dense rank 16 | sparse rank 67 | recovered inside chapter: False
  retrieved: ['hp

### What the six misses have in common

| id | question | gold chunk(s) | dense rank | sparse rank | recovered inside chapter? |
|---|---|---|---|---|---|
| `cand-hp1-01-04` | distinctive feature on Harry's forehead | `hp1-ch01-0013` | 24 | 27 | yes |
| `cand-hp1-06-01` | name of Harry's new owl | `hp1-ch06-0001` | 59 | 4 | yes |
| `cand-hp1-10-03` | spell Ron used to levitate the club | `hp1-ch10-0007` | 12 | 125 | **no** |
| `cand-hp1-11-03` | score of Harry's first match | `hp1-ch11-0010` | 16 | 67 | **no** |
| `cand-hp1-13-03` | spell Malfoy used on Neville | `hp1-ch13-0002` | 12 | 39 | yes |
| `cand-hp1-16-01` | object used to calm Fluffy | `hp1-ch16-0008`, `0009` | 18 | 7 | yes |

All six are `factoid`, and none is a label error in the "chunk does not contain
the answer" sense - each gold chunk does contain it. Three causes:

**1. Cross-chapter dilution (4 of 6).** Scoping retrieval to the question's own
chapter lifts Hit@5 from 59/65 to **63/65 (0.969)**. The gold chunk ranks fine
*within* its chapter; chunks from other chapters crowd it out of the global top 5.
That is not deployable as-is (inference has no chapter label), but it localises
the loss at book scale rather than in local context, which is the argument for a
reranker over a candidate set rather than a better embedding.

**2. Two genuine in-chapter failures.** In both, a single sentence carries the
answer and the question paraphrases it:

- `cand-hp1-10-03` - the gold chunk is the Charms lesson where the incantation is
  taught, not the troll fight the question describes. The spell is never named in
  the troll scene, so this is partly a **Lab 2 label artifact**: the auto-located
  evidence points at the concept's introduction, not the event.
- `cand-hp1-11-03` - "score" versus "won by one hundred and seventy points to
  sixty": no shared term, and the sentence sits two-thirds into the last chunk of
  the chapter.

**3. Dilution, not distance, is the shared symptom.** The answer is mid-to-late
inside a 512-token chunk (up to 83% through `hp1-ch13-0002`), and four of the six
still retrieve chunks from the *correct* chapter - the right scene, wrong
sentence. In `ch10` and `ch13` the gold sits within two chunk positions of what
was returned. That is the strongest evidence in this notebook for **smaller
chunks (256) plus neighbour expansion**, and it should be the first chunking
ablation.

Two secondary observations:

- **Deeper fusion does not help.** Fusing 249 candidates per list instead of 5
  recovers `cand-hp1-16-01` but breaks `cand-hp1-16-02` and `cand-hp1-17-01`
  (58/65): latent candidates accumulate RRF credit from many weak mentions. Keep
  the pool at `top_k` and fix the chunks instead.
- **RRF can lose a sparse-only win.** `cand-hp1-06-01` is sparse rank 4 but dense
  rank 59, so it lands at fused rank 7 while sparse alone would have hit. Fusion
  trades these away for the questions where dense rescues sparse; it is not
  strictly dominant.

Finally, one partial-recall case: `cand-hp1-09-02` (broomstick models) spans
`hp1-ch09-0007` and `hp1-ch09-0008`, but only the first is retrieved. That is why
`Recall@5` (0.900) trails `Hit@5` (0.908).

## Generation evaluation

Retrieval asks whether the evidence reached the prompt; generation asks what the
model did with it. This half scores the answers themselves, so a failure can be
attributed to one stage or the other:

- **`EM` / `F1`** against the gold answer. Cheap and reproducible, but brittle on
  prose - reported on `factoid` rows only.
- **Judge correctness** - does the answer mean the same as the gold answer?
- **Judge faithfulness** - is every claim supported by the retrieved excerpts?
- **Abstention** - does it refuse when the excerpts lack the answer?

The judge is `nova-pro`, deliberately stronger than the `nova-lite` generator, and
returns strict JSON (`correct`, `faithful`, `reason`).


In [10]:
REFUSAL = "I cannot answer that from the provided excerpts."

SYSTEM_TEMPLATE = """You answer questions about the book "{book_name}" using ONLY the excerpts provided in the user's message.

Rules:
1. Base every statement on the provided excerpts. Never use knowledge from outside them, even if you are confident about it.
2. Some excerpts may be irrelevant to the question. Ignore them. Prefer the excerpt that addresses the question most directly.
3. If the excerpts do not contain the answer, reply with exactly this sentence and nothing else:
{refusal}
4. When an excerpt states a concrete action, name it explicitly - the object, the place or the spell - rather than describing the character's general ability.
5. Answer in 1-3 sentences unless the question requires more.
6. After the answer, list the id attribute of each excerpt you relied on, one per line, no other text.
"""

EXCERPT_TEMPLATE = """<excerpt id="[Book {book_num}, Chapter {chapter_num}]" chapter="{chapter_name}">
{text}
</excerpt>"""

BOOK_NAME = CHUNKS[0]["book_name"]


def build_prompt(question, chunk_ids):
    excerpts = "\n\n".join(
        EXCERPT_TEMPLATE.format(
            book_num=CHUNK_BY_ID[cid]["book_num"],
            chapter_num=CHUNK_BY_ID[cid]["chapter_num"],
            chapter_name=CHUNK_BY_ID[cid]["chapter_name"],
            text=CHUNK_BY_ID[cid]["text"],
        )
        for cid in chunk_ids
    )
    system = SYSTEM_TEMPLATE.format(book_name=BOOK_NAME, refusal=REFUSAL)
    return system, f"<excerpts>\n{excerpts}\n</excerpts>\n\nQuestion: {question}\n\nAnswer:"


_CITATION_ONLY = re.compile(r"(?:Book\d+,Chapter\d+)+")


def strip_citations(answer):
    """Drop citation-only lines, including JSON-ish variants, so scoring sees only the answer."""
    kept = []
    for line in answer.splitlines():
        compact = re.sub(r"[\s\[\]\"',]", "", line)
        if compact and _CITATION_ONLY.fullmatch(compact):
            continue
        kept.append(line)
    return "\n".join(kept).strip()


def answer_with(question, chunk_ids):
    system, prompt = build_prompt(question, chunk_ids)
    return bedrock_client.chat(prompt, system=system, temperature=0.0, max_tokens=300)

In [11]:
def normalise_answer(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, gold):
    return float(normalise_answer(prediction) == normalise_answer(gold))


def token_f1(prediction, gold):
    predicted = normalise_answer(prediction).split()
    expected = normalise_answer(gold).split()
    common = collections.Counter(predicted) & collections.Counter(expected)
    overlap = sum(common.values())
    if not overlap:
        return 0.0
    precision = overlap / len(predicted)
    recall = overlap / len(expected)
    return 2 * precision * recall / (precision + recall)


def is_refusal(answer):
    text = answer.strip().lower()
    return text.startswith(REFUSAL.lower()[:30]) or "cannot answer that from the provided" in text


# Sanity check: the examples are intentionally brittle on purpose.
print(exact_match("Grunnings", "Grunnings"), token_f1("Grunnings", "Grunnings"))
print(exact_match("The firm is Grunnings.", "Grunnings"), round(token_f1("The firm is Grunnings.", "Grunnings"), 2))
print(is_refusal(REFUSAL), "|", is_refusal("Grunnings"))

1.0 1.0
0.0 0.5
True | False


In [12]:
JUDGE_TEMPLATE = """You grade a question-answering system that must answer ONLY from the excerpts it was given.

Return strict JSON on one line and nothing else:
{{"correct": true|false, "faithful": true|false, "reason": "<one short sentence>"}}

Definitions:
- correct: the model answer matches the gold answer's meaning. Wording and number format may differ. A refusal is always incorrect here - abstention is scored separately.
- faithful: every factual claim in the model answer is supported by the excerpts.

Question: {question}
Gold answer: {gold}
Model answer: {answer}

Excerpts:
{excerpts}
"""


def parse_json_object(text):
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"judge returned no JSON: {text[:120]!r}")
    return json.loads(text[start:end + 1])


def judge_answer(question, gold, answer, chunk_ids):
    """Ask the stronger model for a strict-JSON verdict on one answer."""
    _, prompt = build_prompt(question, chunk_ids)   # reuse the exact context the generator saw
    excerpts = prompt.split("<excerpts>", 1)[1].split("</excerpts>", 1)[0].strip()
    raw = judge_client.chat(
        JUDGE_TEMPLATE.format(question=question, gold=gold, answer=answer, excerpts=excerpts),
        temperature=0.0,
        max_tokens=300,
    )
    try:
        verdict = parse_json_object(raw)
    except (ValueError, json.JSONDecodeError):
        # Retry once with the JSON requirement restated.
        raw = judge_client.chat(
            "Return ONLY the JSON object.\n\n" + JUDGE_TEMPLATE.format(
                question=question, gold=gold, answer=answer, excerpts=excerpts),
            temperature=0.0,
            max_tokens=200,
        )
        verdict = parse_json_object(raw)
    return {
        "correct": bool(verdict.get("correct")),
        "faithful": bool(verdict.get("faithful")),
        "reason": str(verdict.get("reason", ""))[:300],
    }

In [13]:
GEN_K = 5
GEN_RETRIEVER = "hybrid"

# Six questions whose answers are not in Book 1. Kept famous on purpose: the
# generator has probably memorised them, so a refusal means it obeyed the excerpts.
UNANSWERABLE_PROBES = [
    "Who is Harry's godfather?",
    "What is a Horcrux?",
    "Who is the Half-Blood Prince?",
    "What is the name of Hermione's cat?",
    "Who killed Dumbledore?",
    "What is the village near Hogwarts that students visit on weekends?",
]


def run_generation(record):
    embedding = bedrock_client.embed(record["question"]).reshape(1, -1)
    retrieved = hybrid_ranking(embedding, record["question"], GEN_K)
    answer = strip_citations(answer_with(record["question"], retrieved))
    verdict = judge_answer(record["question"], record["answer"], answer, retrieved)
    retrieval = retrieval_scores(retrieved, record["evidence_chunk_ids"])
    return {
        "id": record["id"],
        "type": record["type"],
        "retriever": GEN_RETRIEVER,
        "chunk_size": INDEX_CONFIG["chunk_size"],
        "top_k": GEN_K,
        "hit@k": retrieval["hit"],
        "recall@k": retrieval["recall"],
        "mrr": retrieval["rr"],
        "em": exact_match(answer, record["answer"]),
        "f1": token_f1(answer, record["answer"]),
        "judge_correct": verdict["correct"],
        "judge_faithful": verdict["faithful"],
        "judge_reason": verdict["reason"],
        "abstained": is_refusal(answer),
        "question": record["question"],
        "gold": record["answer"],
        "answer": answer,
        "retrieved_ids": "|".join(retrieved),
    }


In [14]:
GEN_ROWS = [run_generation(r) for r in tqdm(usable, desc="generate + judge")]

for probe in UNANSWERABLE_PROBES:
    embedding = bedrock_client.embed(probe).reshape(1, -1)
    retrieved = hybrid_ranking(embedding, probe, GEN_K)
    answer = strip_citations(answer_with(probe, retrieved))
    GEN_ROWS.append({
        "id": f"probe-{len([r for r in GEN_ROWS if r['type'] == 'unanswerable']) + 1:02d}",
        "type": "unanswerable",
        "retriever": GEN_RETRIEVER,
        "chunk_size": INDEX_CONFIG["chunk_size"],
        "top_k": GEN_K,
        "hit@k": "", "recall@k": "", "mrr": "", "em": "", "f1": "",
        "judge_correct": "", "judge_faithful": "", "judge_reason": "",
        "abstained": is_refusal(answer),
        "question": probe, "gold": "", "answer": answer,
        "retrieved_ids": "|".join(retrieved),
    })

answerable = [r for r in GEN_ROWS if r["type"] != "unanswerable"]
probes = [r for r in GEN_ROWS if r["type"] == "unanswerable"]
factoid = [r for r in answerable if r["type"] == "factoid"]

print(f"\nanswerable: {len(answerable)} | probes: {len(probes)}")
print(f"judge correct  : {sum(r['judge_correct'] for r in answerable) / len(answerable):.3f}")
print(f"judge faithful : {sum(r['judge_faithful'] for r in answerable) / len(answerable):.3f}")
refused = [r for r in answerable if r["abstained"]]
over_abstain = [r for r in refused if r["hit@k"]]
guarded = [r for r in refused if not r["hit@k"]]
print(f"refusals       : {len(refused)}/{len(answerable)}"
      f" | over-abstention (evidence was retrieved): {len(over_abstain)}"
      f" | correct refusal (retrieval miss): {len(guarded)}")
print(f"EM / F1 (factoid only): {sum(r['em'] for r in factoid) / len(factoid):.3f} / "
      f"{sum(r['f1'] for r in factoid) / len(factoid):.3f}")
print(f"true abstain (probes): {sum(r['abstained'] for r in probes) / len(probes):.3f}"
      f" ({sum(r['abstained'] for r in probes)}/{len(probes)})")

generate + judge:   0%|          | 0/65 [00:00<?, ?it/s]

generate + judge:   2%|▏         | 1/65 [00:01<01:55,  1.81s/it]

generate + judge:   3%|▎         | 2/65 [00:03<01:42,  1.62s/it]

generate + judge:   5%|▍         | 3/65 [00:04<01:33,  1.51s/it]

generate + judge:   6%|▌         | 4/65 [00:06<01:33,  1.54s/it]

...

generate + judge:  97%|█████████▋| 63/65 [01:39<00:03,  1.62s/it]

generate + judge:  98%|█████████▊| 64/65 [01:41<00:01,  1.61s/it]

generate + judge: 100%|██████████| 65/65 [01:43<00:00,  1.66s/it]

generate + judge: 100%|██████████| 65/65 [01:43<00:00,  1.59s/it]


answerable: 65 | probes: 6
judge correct  : 0.877
judge faithful : 0.954
refusals       : 3/65 | over-abstention (evidence was retrieved): 1 | correct refusal (retrieval miss): 2
EM / F1 (factoid only): 0.000 / 0.222
true abstain (probes): 1.000 (6/6)


In [15]:
# The point of scoring both stages: attribute each failure to one of them.
crosstab = collections.Counter(
    ("retrieval hit" if r["hit@k"] else "retrieval miss",
     "judge correct" if r["judge_correct"] else "judge wrong")
    for r in answerable
)
print("\nretrieval x generation (answerable):")
for key in sorted(crosstab):
    print(f"  {key[0]:<15} + {key[1]:<14}: {crosstab[key]:>2}")

RESULTS_FIELDS = [
    "id", "type", "retriever", "chunk_size", "top_k", "hit@k", "recall@k", "mrr",
    "em", "f1", "judge_correct", "judge_faithful", "judge_reason", "abstained",
    "question", "gold", "answer", "retrieved_ids",
]
out_path = LABS / "eval" / "results.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(GEN_ROWS)
print(f"\nwrote {len(GEN_ROWS)} rows -> {out_path}")


retrieval x generation (answerable):
  retrieval hit   + judge correct : 53
  retrieval hit   + judge wrong   :  6
  retrieval miss  + judge correct :  4
  retrieval miss  + judge wrong   :  2

wrote 71 rows -> /Users/tc/Documents/Github/monitus_code/labs/eval/results.csv


In [16]:
# Hand-check a mix of correct and incorrect verdicts once, before trusting the judge.
wrong = [r for r in answerable if not r["judge_correct"]]
right = [r for r in answerable if r["judge_correct"]]
for r in (wrong[:5] + right[:5]):
    print(f"\n[{r['id']}] {r['question']}")
    print(f"  gold    : {r['gold']}")
    print(f"  answer  : {r['answer'][:220]}")
    print(f"  verdict : correct={r['judge_correct']} faithful={r['judge_faithful']}")
    print(f"  reason  : {r['judge_reason']}")


[cand-hp1-02-03] How many birthday presents did Dudley receive before his parents promised to buy him two more?
  gold    : thirty-seven
  answer  : Dudley had thirty-six birthday presents before his parents promised to buy him two more.

["[Book 1, Chapter 2]" chapter="Chapter 2 - The Vanishing Glass": ". “Thirty-six,” he said, looking up at his mother and father. “
  verdict : correct=False faithful=True
  reason  : The model answer incorrectly states Dudley had thirty-six presents initially, while the excerpt confirms he had thirty-seven before his parents promised two more.

[cand-hp1-05-03] What are the two magical substances mentioned as guarding the high-security vaults at Gringotts?
  gold    : spells, dragons
  answer  : The two magical substances mentioned as guarding the high-security vaults at Gringotts are spells and enchantments, and dragons.

- The text states, "Why would you be mad to try and rob Gringotts? Spells — enchantments,"
  verdict : correct=False faithful=Tru

### What generation adds

| metric | value |
|---|---|
| judge correctness | **0.877** (57/65) |
| judge faithfulness | **0.954** (62/65) |
| `EM` (factoid) | 0.000 |
| `F1` (factoid) | 0.222 |
| over-abstention (evidence retrieved) | 1/65 |
| guardrail refusals (retrieval missed) | 2/65 |
| true abstention on probes | 6/6 |

**The cross-tab is the result.**

| | judge correct | judge wrong |
|---|---|---|
| retrieval hit (59) | 53 | 6 |
| retrieval miss (6) | 4 | 2 |

- **53 clean successes** - the evidence was retrieved and the answer used it.
- **4 of 6 retrieval misses were answered correctly from memory.** Those are the
  parametric-leak cells: the gold chunk never reached the prompt, yet the model
  produced the right answer. End-to-end correctness therefore *overstates* RAG
  quality - on those four the pipeline was right for the wrong reason.
- **Three refusals:** two after a retrieval miss (the guardrail working) and one
  **over-abstention** (`13-04`) where the evidence was retrieved but the model
  judged the excerpts insufficient.
- **Five answer errors, all local details:** `02-03` counting, `05-03` adds
  "enchantments" the excerpt contains, `11-04` omits "bright blue flames",
  `14-03` says McGonagall instead of Snape (the only unfaithful answer), and
  `15-02` reports one student's 50 instead of the 150 total.

**`EM` is useless here, and that is expected.** The generator answers in prose, so
exact match fires on 0 of 62 factoid rows; `F1 = 0.222`. Keep them as a sanity
signal and rank configurations with the judge.

**Abstention works on the probes.** All six questions outside Book 1 were refused,
including trivia the model certainly knows, so the grounding instruction holds.
Six probes are a smoke test, not a rate.

**Generation is noisy.** Two identical temperature-0 runs changed 58 of 65
answers, moving correctness between 0.877 and 0.892 and faithfulness between 0.95
and 0.99. Trust retrieval deltas; treat one- or two-row generation differences as
noise (see *Reproducibility* at the end).

## Reranker ablation - hybrid top-50 -> rerank -> top-5

The failure analysis showed that 4 of the 6 retrieval misses were not local
failures: the gold chunk was in the right neighbourhood but lost to chunks from
other chapters. That is the case a reranker is built for - reorder a fixed
candidate pool instead of changing the index.

The ceiling is easy to check. Fusing a **top-50** pool from dense and sparse
contains **all 65 gold chunks** (candidate `Hit@50 = 1.000`), so any remaining
miss after reranking is the reranker's fault, not recall's.

- Reranker: **`amazon.rerank-v1:0`** through `bedrock-agent-runtime.rerank`.
  Cohere Rerank 3.5 is available in the catalogue but this account cannot invoke
  it (AWS Marketplace eligibility), so Amazon is the fallback. Its relevance
  scores are near-zero but correctly ordered; only the order is used.
- One call per question: 50 chunks, ~96 billed documents (512-token chunks split
  at the query+document limit), still one query unit - about **$0.13 for 65
  questions** at Cohere-like rates.
- Results are cached in `data/rerank_cache.jsonl`, so re-runs cost nothing and
  return the same ranking.

In [17]:
RERANK_MODEL = "amazon.rerank-v1:0"
RERANK_POOL = 50
RERANK_TOP_K = 5
RERANK_MODEL_ARN = f"arn:aws:bedrock:{bedrock_client.region}::foundation-model/{RERANK_MODEL}"
RERANK_CACHE_PATH = DATA / "rerank_cache.jsonl"

rerank_client = boto3.client(
    "bedrock-agent-runtime",
    region_name=bedrock_client.region,
    config=Config(read_timeout=120, retries={"max_attempts": 5}),
)

RERANK_CACHE = {}
if RERANK_CACHE_PATH.exists():
    for line in RERANK_CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            RERANK_CACHE[record["key"]] = record["ranking"]


def _rerank_key(question, chunk_ids, top_n):
    raw = f"{RERANK_MODEL}|{question}|{top_n}|{'|'.join(chunk_ids)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def rerank_chunks(question, chunk_ids, top_n=RERANK_TOP_K):
    """Reorder candidate chunk ids with the Bedrock reranker (disk-cached)."""
    key = _rerank_key(question, chunk_ids, top_n)
    if key in RERANK_CACHE:
        return RERANK_CACHE[key]

    response = rerank_client.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": question}}],
        sources=[
            {"type": "INLINE", "inlineDocumentSource": {
                "type": "TEXT", "textDocument": {"text": CHUNK_BY_ID[cid]["text"]}}}
            for cid in chunk_ids
        ],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "modelConfiguration": {"modelArn": RERANK_MODEL_ARN},
                "numberOfResults": top_n,
            },
        },
    )
    ranking = [
        {"chunk_id": chunk_ids[item["index"]], "score": item["relevanceScore"]}
        for item in response["results"]
    ]
    RERANK_CACHE[key] = ranking
    with open(RERANK_CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "ranking": ranking}) + "\n")
    return ranking


def rerank_retrieve(question, pool=RERANK_POOL, top_n=RERANK_TOP_K):
    embedding = bedrock_client.embed(question).reshape(1, -1)
    candidates = hybrid_ranking(embedding, question, pool)
    return [r["chunk_id"] for r in rerank_chunks(question, candidates, top_n)], candidates


smoke_ids, smoke_candidates = rerank_retrieve(usable[0]["question"])
print(f"pool {len(smoke_candidates)} -> reranked {len(smoke_ids)}: {smoke_ids}")
print("expected:", usable[0]["evidence_chunk_ids"])

pool 50 -> reranked 5: ['hp1-ch01-0001', 'hp1-ch05-0017', 'hp1-ch01-0003', 'hp1-ch01-0005', 'hp1-ch02-0009']
expected: ['hp1-ch01-0001']


In [18]:
RERANKED = {}
comparison = []
for record in tqdm(usable, desc="rerank retrieval"):
    relevant = record["evidence_chunk_ids"]
    baseline = hybrid_ranking(
        bedrock_client.embed(record["question"]).reshape(1, -1), record["question"], 5
    )
    reranked, candidates = rerank_retrieve(record["question"])
    RERANKED[record["id"]] = reranked
    comparison.append({
        "id": record["id"],
        "baseline": retrieval_scores(baseline, relevant),
        "pool_top5": retrieval_scores(candidates[:5], relevant),
        "reranked": retrieval_scores(reranked, relevant),
        "candidate_hit": bool(set(candidates) & set(relevant)),
    })


def mean_metrics(key):
    n = len(comparison)
    return (
        sum(r[key]["hit"] for r in comparison) / n,
        sum(r[key]["recall"] for r in comparison) / n,
        sum(r[key]["rr"] for r in comparison) / n,
    )


print(f"candidate Hit@{RERANK_POOL}: "
      f"{sum(r['candidate_hit'] for r in comparison)}/{len(comparison)}")
for label, key in (("hybrid@5", "baseline"),
                   (f"pool{RERANK_POOL} -> top5", "pool_top5"),
                   (f"pool{RERANK_POOL} -> rerank@5", "reranked")):
    hit, recall, mrr = mean_metrics(key)
    print(f"{label:<22} hit={hit:.3f} recall={recall:.3f} mrr={mrr:.3f}")

base_hits = {r["id"] for r in comparison if r["baseline"]["hit"]}
reranked_hits = {r["id"] for r in comparison if r["reranked"]["hit"]}
print("fixed :", sorted(reranked_hits - base_hits))
print("lost  :", sorted(base_hits - reranked_hits))
print("still missing:", sorted({r["id"] for r in comparison} - reranked_hits))

rerank retrieval:   0%|          | 0/65 [00:00<?, ?it/s]

rerank retrieval:   2%|▏         | 1/65 [00:00<00:50,  1.26it/s]

rerank retrieval:   3%|▎         | 2/65 [00:02<01:38,  1.57s/it]

rerank retrieval:   5%|▍         | 3/65 [00:03<01:15,  1.21s/it]

rerank retrieval:   6%|▌         | 4/65 [00:04<01:02,  1.03s/it]

...

rerank retrieval:  97%|█████████▋| 63/65 [01:02<00:02,  1.20s/it]

rerank retrieval:  98%|█████████▊| 64/65 [01:03<00:01,  1.08s/it]

rerank retrieval: 100%|██████████| 65/65 [01:04<00:00,  1.00s/it]

rerank retrieval: 100%|██████████| 65/65 [01:04<00:00,  1.01it/s]

candidate Hit@50: 65/65
hybrid@5               hit=0.908 recall=0.900 mrr=0.742
pool50 -> top5         hit=0.892 recall=0.869 mrr=0.736
pool50 -> rerank@5     hit=0.969 recall=0.962 mrr=0.882
fixed : ['cand-hp1-06-01', 'cand-hp1-11-03', 'cand-hp1-13-03', 'cand-hp1-16-01']
lost  : []
still missing: ['cand-hp1-01-04', 'cand-hp1-10-03']


In [19]:
RERANK_GEN_ROWS = []
for record in tqdm(usable, desc="reranked generation"):
    chunk_ids = RERANKED[record["id"]]
    answer = strip_citations(answer_with(record["question"], chunk_ids))
    verdict = judge_answer(record["question"], record["answer"], answer, chunk_ids)
    retrieval = retrieval_scores(chunk_ids, record["evidence_chunk_ids"])
    RERANK_GEN_ROWS.append({
        "id": record["id"],
        "type": record["type"],
        "retriever": "hybrid+rerank",
        "chunk_size": INDEX_CONFIG["chunk_size"],
        "top_k": RERANK_TOP_K,
        "hit@k": retrieval["hit"],
        "recall@k": retrieval["recall"],
        "mrr": retrieval["rr"],
        "em": exact_match(answer, record["answer"]),
        "f1": token_f1(answer, record["answer"]),
        "judge_correct": verdict["correct"],
        "judge_faithful": verdict["faithful"],
        "judge_reason": verdict["reason"],
        "abstained": is_refusal(answer),
        "question": record["question"],
        "gold": record["answer"],
        "answer": answer,
        "retrieved_ids": "|".join(chunk_ids),
    })

n = len(RERANK_GEN_ROWS)
base_answerable = [r for r in GEN_ROWS if r["type"] != "unanswerable"]
print(f"baseline : correct {sum(r['judge_correct'] for r in base_answerable) / len(base_answerable):.3f}"
      f" | faithful {sum(r['judge_faithful'] for r in base_answerable) / len(base_answerable):.3f}")
print(f"reranked : correct {sum(r['judge_correct'] for r in RERANK_GEN_ROWS) / n:.3f}"
      f" | faithful {sum(r['judge_faithful'] for r in RERANK_GEN_ROWS) / n:.3f}")
print(f"false abstain (reranked): {sum(r['abstained'] for r in RERANK_GEN_ROWS) / n:.3f}")

base_correct = {r["id"]: r["judge_correct"] for r in base_answerable}
print("flipped to correct:", sorted(r["id"] for r in RERANK_GEN_ROWS
                                     if r["judge_correct"] and not base_correct[r["id"]]))
print("flipped to wrong  :", sorted(r["id"] for r in RERANK_GEN_ROWS
                                     if not r["judge_correct"] and base_correct[r["id"]]))

with open(LABS / "eval" / "results_rerank.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(RERANK_GEN_ROWS)
print(f"wrote {len(RERANK_GEN_ROWS)} rows -> eval/results_rerank.csv")

reranked generation:   0%|          | 0/65 [00:00<?, ?it/s]

reranked generation:   2%|▏         | 1/65 [00:01<01:14,  1.16s/it]

reranked generation:   3%|▎         | 2/65 [00:02<01:11,  1.14s/it]

reranked generation:   5%|▍         | 3/65 [00:03<01:08,  1.11s/it]

reranked generation:   6%|▌         | 4/65 [00:04<01:11,  1.17s/it]

...

reranked generation:  97%|█████████▋| 63/65 [01:13<00:02,  1.15s/it]

reranked generation:  98%|█████████▊| 64/65 [01:15<00:01,  1.23s/it]

reranked generation: 100%|██████████| 65/65 [01:16<00:00,  1.22s/it]

reranked generation: 100%|██████████| 65/65 [01:16<00:00,  1.17s/it]

baseline : correct 0.877 | faithful 0.954
reranked : correct 0.938 | faithful 0.985
false abstain (reranked): 0.000
flipped to correct: ['cand-hp1-05-03', 'cand-hp1-06-01', 'cand-hp1-11-03', 'cand-hp1-11-04', 'cand-hp1-15-02']
flipped to wrong  : ['cand-hp1-03-04']
wrote 65 rows -> eval/results_rerank.csv


### What the reranker bought

**Retrieval** (deterministic):

| stage | Hit@5 | Recall@5 | MRR |
|---|---|---|---|
| hybrid@5 (baseline) | 0.908 | 0.900 | 0.742 |
| pool 50 -> top 5 (no rerank) | 0.892 | 0.869 | 0.736 |
| pool 50 -> rerank -> top 5 | **0.969** | **0.962** | **0.882** |

- **Candidate `Hit@50 = 65/65`.** Every gold chunk was already in the pool, so the
  gain is pure ordering - there is nothing left for recall to fix.
- **The deeper pool alone is slightly worse** (0.892 vs 0.908): RRF over 50
  candidates promotes chunks that appear mid-list in both retrievers. The +4
  recovered questions come from the reranker, not from widening the pool.
- **4 misses fixed** (`06-01`, `11-03`, `13-03`, `16-01`), **none broken**. Two
  remain: `01-04` (the scar) and `10-03` (the incantation).

**Generation** on the reranked top-5:

| | baseline | reranked |
|---|---|---|
| judge correctness | 0.877 (57/65) | **0.938** (61/65) |
| judge faithfulness | 0.954 (62/65) | 0.985 (64/65) |
| refusals on answerable | 3/65 | **0/65** |

- **+6.1 points of correctness** for one cheap, cached reorder, and the reranked
  score (61/65) held across two runs despite the generator's noise.
- Five answers flipped to correct: `05-03`, `06-01`, `11-03`, `11-04`, `15-02`.
  The first two were refusals after a retrieval miss - once the evidence was in
  the prompt, the model answered.
- **No refusals left.** Every question now has enough context to answer.
- **One regression:** `03-04` answered "a gloomy-looking hotel" instead of
  "Railview Hotel" - the reranker moved the descriptive chunk above the naming one.
- `13-04` changed character rather than improving: it used to refuse, now it
  confidently answers "Madam Hooch". The reranked context contains both referees
  (Hooch in ch11, Snape in ch13), so this is the **distractor** failure from the
  taxonomy - the wrong match, not the wrong passage.

**Remaining errors after reranking:** `02-03`, `03-04`, `13-04`, `14-03` - all
answer-side. Retrieval is no longer the bottleneck on this set.

**Cost and reproducibility.** One rerank call per question (50 chunks, ~96 billed
documents = one query unit), about **$0.13 per 65-question pass**. Rankings are
cached in `data/rerank_cache.jsonl`, so re-runs are free and identical.

## Closed-book control

Every number so far can be inflated by a model that already knows the book. This
section removes retrieval entirely and asks the same 65 questions from memory, so
the two sources can be separated:

- **closed-book correctness** - what the model knows without the text.
- **retrieval-augmented correctness** - the reranked pipeline from above.
- the **cross-tab** - how many questions context flips from wrong to right, and
  how many it flips the other way.

The generator and judge prompts are the only things that change; the model is the
same `nova-lite`, the judge the same `nova-pro`.

In [20]:
CLOSED_BOOK_SYSTEM = """You answer questions about the Harry Potter series from your own knowledge. No excerpts are provided.

If you do not know the answer, reply with exactly: I don't know.
Otherwise answer in 1-3 sentences and nothing else."""

CLOSED_BOOK_JUDGE = """You grade a question-answering system that answers from memory, with no excerpts.

Return strict JSON on one line and nothing else:
{{"correct": true|false, "reason": "<one short sentence>"}}

correct: the model answer matches the gold answer's meaning. Wording and number format may differ. "I don't know" or any refusal is incorrect.

Question: {question}
Gold answer: {gold}
Model answer: {answer}
"""


def answer_closed_book(question):
    return bedrock_client.chat(
        question, system=CLOSED_BOOK_SYSTEM, temperature=0.0, max_tokens=200
    ).strip()


def judge_closed_book(question, gold, answer):
    raw = judge_client.chat(
        CLOSED_BOOK_JUDGE.format(question=question, gold=gold, answer=answer),
        temperature=0.0,
        max_tokens=200,
    )
    verdict = parse_json_object(raw)
    return bool(verdict.get("correct")), str(verdict.get("reason", ""))[:300]


CLOSED_BOOK_ROWS = []
for record in tqdm(usable, desc="closed-book"):
    answer = answer_closed_book(record["question"])
    correct, reason = judge_closed_book(record["question"], record["answer"], answer)
    CLOSED_BOOK_ROWS.append({
        "id": record["id"], "type": record["type"], "retriever": "closed-book",
        "chunk_size": "", "top_k": 0, "hit@k": "", "recall@k": "", "mrr": "",
        "em": exact_match(answer, record["answer"]),
        "f1": token_f1(answer, record["answer"]),
        "judge_correct": correct, "judge_faithful": "", "judge_reason": reason,
        "abstained": answer.startswith("I don't know"),
        "question": record["question"], "gold": record["answer"], "answer": answer,
        "retrieved_ids": "",
    })

n = len(CLOSED_BOOK_ROWS)
closed_book = {r["id"]: r for r in CLOSED_BOOK_ROWS}
augmented = {r["id"]: r for r in RERANK_GEN_ROWS}

print(f"closed-book correct: {sum(r['judge_correct'] for r in CLOSED_BOOK_ROWS) / n:.3f}"
      f" ({sum(r['judge_correct'] for r in CLOSED_BOOK_ROWS)}/{n})")
print(f"closed-book EM/F1  : {sum(r['em'] for r in CLOSED_BOOK_ROWS) / n:.3f} / "
      f"{sum(r['f1'] for r in CLOSED_BOOK_ROWS) / n:.3f}")
print(f"'I don't know'     : {sum(r['abstained'] for r in CLOSED_BOOK_ROWS)}/{n}")

book_context_crosstab = collections.Counter(
    ("closed-book correct" if closed_book[r["id"]]["judge_correct"] else "closed-book wrong",
     "context correct" if augmented[r["id"]]["judge_correct"] else "context wrong")
    for r in usable
)
print("\nclosed-book x reranked-context:")
for key in sorted(book_context_crosstab):
    print(f"  {key[0]:<20} + {key[1]:<16}: {book_context_crosstab[key]:>2}")

with open(LABS / "eval" / "results_closed_book.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(CLOSED_BOOK_ROWS)
print(f"wrote {len(CLOSED_BOOK_ROWS)} rows -> eval/results_closed_book.csv")

closed-book:   0%|          | 0/65 [00:00<?, ?it/s]

closed-book:   2%|▏         | 1/65 [00:00<01:02,  1.03it/s]

closed-book:   3%|▎         | 2/65 [00:02<01:03,  1.02s/it]

closed-book:   5%|▍         | 3/65 [00:03<01:18,  1.26s/it]

closed-book:   6%|▌         | 4/65 [00:04<01:13,  1.21s/it]

...

closed-book:  97%|█████████▋| 63/65 [01:15<00:03,  1.55s/it]

closed-book:  98%|█████████▊| 64/65 [01:16<00:01,  1.43s/it]

closed-book: 100%|██████████| 65/65 [01:17<00:00,  1.39s/it]

closed-book: 100%|██████████| 65/65 [01:17<00:00,  1.19s/it]

closed-book correct: 0.415 (27/65)
closed-book EM/F1  : 0.000 / 0.087
'I don't know'     : 3/65

closed-book x reranked-context:
  closed-book correct  + context correct : 25
  closed-book correct  + context wrong   :  2
  closed-book wrong    + context correct : 36
  closed-book wrong    + context wrong   :  2
wrote 65 rows -> eval/results_closed_book.csv


### What the closed-book control shows

| | value |
|---|---|
| closed-book correctness | 0.415 (27/65) |
| retrieval-augmented correctness (reranked) | 0.938 (61/65) |
| closed-book `EM` / `F1` | 0.000 / 0.087 |
| "I don't know" | 3/65 |

| | context correct | context wrong |
|---|---|---|
| closed-book correct | 25 | 2 |
| closed-book wrong | 36 | 2 |

- **Memory alone answers roughly 40-50% of the set** - 27/65 (0.415) this run, 32/65
  (0.492) on an earlier identical run. The spread is the generator's noise, not the
  method, and it is exactly why the augmented score needed this control.
- **Context flips 36 questions** from wrong without it to right with it. That is
  the value the pipeline actually adds, and it is large.
- **Context almost never hurts:** only 2 questions were right closed-book and
  wrong with context.
- **The model rarely abstains** - 3 of 65 said "I don't know" from memory. It
  guesses fluently, so a closed-book score understates its willingness to be wrong.
- **Read 0.938 as real retrieval work, not recitation.**

**Caveat.** A closed-book control proves that memory is *insufficient*; it does
not prove the model is *reading* the excerpts it is given - it could ignore them
and answer from memory. Proving that needs a **placebo** index (irrelevant chunks)
or the perturbation probes of Milestone 5, where the context is deliberately
wrong and the answer reveals which source was trusted.

### Reproducibility

- **Retrieval is deterministic.** Embeddings are cached, Chroma and BM25 are
  exact, and the reranker rankings are cached, so re-running reproduces
  `Hit@5 = 0.969`, `Recall@5 = 0.962`, `MRR = 0.882` to the digit.
- **Generation is not.** Two identical runs at temperature 0 changed 58 of 65
  answers. Across runs, baseline correctness moved 0.877-0.892, faithfulness
  0.954-0.985, and closed-book correctness 0.415-0.492.
- **Consequences.** Trust retrieval deltas; treat generation differences of one
  or two rows as noise; the reranked correctness (61/65) did hold across both
  runs.
- **Fix.** Cache generator and judge outputs keyed by `(model, prompt)`, exactly
  like the embedding and rerank caches, so a published run is reproducible even
  though the model is stochastic.